# Lab 07 - Cross Validation: Impact of AI on Students

        Source dataset: `Datasets/Impact of AI on Students/ai_student_impact_dataset.csv`

        This notebook adapts the class lab pattern to the student-impact dataset. The source file is never modified.

        ## Lab concepts used

        - Compare one holdout result with repeated shuffled folds.
- Use stratification for classification.
- Report mean and variability rather than one score.

        Interpretation is predictive and associative only. The Kaggle source does not document how the records were collected or whether they represent observed students.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

def find_dataset():
    relative = Path("Datasets/Impact of AI on Students/ai_student_impact_dataset.csv")
    for start in [Path.cwd(), *Path.cwd().parents]:
        candidate = start / relative
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {relative} from {Path.cwd()}")

DATA_PATH = find_dataset()
df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns from {DATA_PATH}")

In [ ]:
IDENTIFIER = "Student_ID"
OUTCOMES = ["Post_Semester_GPA", "Skill_Retention_Score", "Burnout_Risk_Level"]
EARLY_RISK_FEATURES = [
    "Major_Category", "Year_of_Study", "Pre_Semester_GPA",
    "Weekly_GenAI_Hours", "Primary_Use_Case",
    "Prompt_Engineering_Skill", "Tool_Diversity", "Paid_Subscription",
    "Traditional_Study_Hours", "Perceived_AI_Dependency",
    "Institutional_Policy",
]
EXPANDED_FEATURES = EARLY_RISK_FEATURES + ["Anxiety_Level_During_Exams"]

df["GPA_Change"] = df["Post_Semester_GPA"] - df["Pre_Semester_GPA"]
df["GPA_Declined"] = (df["GPA_Change"] < 0).astype(int)

assert IDENTIFIER not in EARLY_RISK_FEATURES
assert not set(OUTCOMES).intersection(EARLY_RISK_FEATURES)
print("Leakage policy ready. Primary burnout model excludes anxiety and all post-semester outcomes.")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

def make_preprocessor(frame, scale_numeric=True):
    categorical = [
        column for column in frame.columns
        if pd.api.types.is_string_dtype(frame[column])
        or pd.api.types.is_bool_dtype(frame[column])
    ]
    numeric = [column for column in frame.columns if column not in categorical]
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(numeric_steps), numeric),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]),
                categorical,
            ),
        ]
    )

In [ ]:
from sklearn.model_selection import RepeatedKFold, RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LogisticRegression

X_reg = df[EXPANDED_FEATURES]
y_reg = df["Skill_Retention_Score"]
reg_pipe = Pipeline([
    ("preprocess", make_preprocessor(X_reg)),
    ("model", Ridge(alpha=1.0)),
])
reg_cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
reg_scores = cross_validate(
    reg_pipe, X_reg, y_reg, cv=reg_cv,
    scoring={"MAE": "neg_mean_absolute_error", "R2": "r2"},
    n_jobs=-1,
)

X_cls = df[EARLY_RISK_FEATURES]
y_cls = df["Burnout_Risk_Level"]
cls_pipe = Pipeline([
    ("preprocess", make_preprocessor(X_cls)),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
cls_cv = RepeatedStratifiedKFold(
    n_splits=5, n_repeats=2, random_state=RANDOM_STATE
)
cls_scores = cross_validate(
    cls_pipe, X_cls, y_cls, cv=cls_cv,
    scoring={"Macro F1": "f1_macro", "Balanced accuracy": "balanced_accuracy"},
    n_jobs=-1,
)

summary = pd.DataFrame({
    "metric": ["Regression MAE", "Regression R2", "Classification macro-F1", "Classification balanced accuracy"],
    "mean": [
        -reg_scores["test_MAE"].mean(),
        reg_scores["test_R2"].mean(),
        cls_scores["test_Macro F1"].mean(),
        cls_scores["test_Balanced accuracy"].mean(),
    ],
    "std": [
        reg_scores["test_MAE"].std(ddof=1),
        reg_scores["test_R2"].std(ddof=1),
        cls_scores["test_Macro F1"].std(ddof=1),
        cls_scores["test_Balanced accuracy"].std(ddof=1),
    ],
})
display(summary)

## What was learned from Lab 7

There is no temporal or repeated-student structure, so chronological splitting would be unjustified. Repeated shuffled folds measure sampling variability while stratification preserves burnout-class proportions.